In [1]:
import random
import torch
import os
import numpy as np
import pandas as pd
import polars as pl

In [2]:
INPUT_DIR = '.'

In [3]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [4]:
def make_aggregated_outputs(input_file, metrics = ['accuracy', 'precision', 'recall', 'f1', 'kappa', 'MCC']):
    dimensions = [
        'informational_vs_involved',
        'non-narrative_vs_narrative',
        'situation-dependent_vs_explicit',
        'non-persuasive_vs_persuasive',
        'non-abstract_vs_abstract',
        'compressed_vs_elaborated'
    ]

    saved_output_dict = {}

    for folder in os.listdir(INPUT_DIR):
        folder_path = os.path.join(INPUT_DIR, folder)

        if not (os.path.isdir(folder_path) and 'outputs' in folder):
            continue

        saved_output_dict[folder] = {
            dim: {metric: [] for metric in metrics}
            for dim in dimensions
        }

        for sub_folder in os.listdir(folder_path):
            sub_path = os.path.join(folder_path, sub_folder)

            if not (os.path.isdir(sub_path) and sub_folder.isdigit()):
                continue

            file_path = os.path.join(sub_path, f"{input_file}.csv")
            df = pd.read_csv(file_path)

            for dimension in dimensions:
                temp_df = df[df['dimension'] == dimension]

                if temp_df.empty:
                    raise ValueError(f"There should be something in the temp_df for the dimension {dimension}.")

                row = temp_df.iloc[0]

                for metric in metrics:
                    saved_output_dict[folder][dimension][metric].append(float(row[metric]))

    temp_dict_all = {}

    for folder, dimensions_dict in saved_output_dict.items():
        series_list = []

        for dimension, metrics_dict in dimensions_dict.items():
            mean_series = pd.Series({
                metric: (sum(values) / len(values)) if values else float('nan')
                for metric, values in metrics_dict.items()
            }, name=dimension)

            series_list.append(mean_series)

        df_folder = pd.concat(series_list, axis=1)
        temp_dict_all[folder] = df_folder

    df_all_folders = pd.concat(temp_dict_all, axis=0)

    df_mean_all = df_all_folders.groupby(level=1).mean()

    return df_all_folders, df_mean_all

In [5]:
all_classif, mean_classif = make_aggregated_outputs('classification_comparison_results_zero_vs_biber')

In [6]:
all_classif

informational_vs_involved  non-narrative_vs_narrative  \
outputsTrain accuracy                    0.633800                    0.477000   
             precision                   0.549002                    0.371242   
             recall                      0.650073                    0.433327   
             f1                          0.594319                    0.399023   
             kappa                       0.265384                   -0.058759   
             MCC                         0.269069                   -0.059750   
outputsTest  accuracy                    0.639300                    0.481100   
             precision                   0.552493                    0.372626   
             recall                      0.656467                    0.438636   
             f1                          0.599238                    0.401769   
             kappa                       0.275671                   -0.050567   
             MCC                         0.279726                   -0.051685   
outputsAll   accuracy                    0.636600                    0.489900   
             precision                   0.552382                    0.384777   
             recall                      0.649411                    0.449461   
             f1                          0.596096                    0.413469   
             kappa                       0.269666                   -0.032580   
             MCC                         0.273217                   -0.033148   

                        situation-dependent_vs_explicit  \
outputsTrain accuracy                          0.532400   
             precision                         0.582000   
             recall                            0.683015   
             f1                                0.627611   
             kappa                             0.009203   
             MCC                               0.009804   
outputsTest  accuracy                          0.527100   
             precision                         0.577348   
             recall                            0.685140   
             f1                                0.625604   
             kappa                            -0.004307   
             MCC                              -0.004414   
outputsAll   accuracy                          0.533400   
             precision                         0.581974   
             recall                            0.681829   
             f1                                0.627059   
             kappa                             0.012720   
             MCC                               0.013270   

                        non-persuasive_vs_persuasive  \
outputsTrain accuracy                       0.566400   
             precision                      0.425452   
             recall                         0.657350   
             f1                             0.515256   
             kappa                          0.155862   
             MCC                            0.168218   
outputsTest  accuracy                       0.572400   
             precision                      0.434461   
             recall                         0.667913   
             f1                             0.525370   
             kappa                          0.166956   
             MCC                            0.180518   
outputsAll   accuracy                       0.562100   
             precision                      0.418882   
             recall                         0.656404   
             f1                             0.510182   
             kappa                          0.148592   
             MCC                            0.161648   

                        non-abstract_vs_abstract  compressed_vs_elaborated  
outputsTrain accuracy                   0.439600                  0.503000  
             precision                  0.230326                  0.166804  
             recall                     0.681620                  

In [7]:
mean_classif

,informational_vs_involved,non-narrative_vs_narrative,situation-dependent_vs_explicit,non-persuasive_vs_persuasive,non-abstract_vs_abstract,compressed_vs_elaborated
MCC,0.274004,-0.048194,0.006220,0.170128,0.035501,-0.059762
accuracy,0.636567,0.482667,0.530967,0.566967,0.438700,0.499467
f1,0.596551,0.404754,0.626758,0.516936,0.337992,0.229106
kappa,0.270240,-0.047302,0.005872,0.157137,0.025012,-0.048682
precision,0.551292,0.376215,0.580440,0.426265,0.227739,0.161529
recall,0.651984,0.440475,0.683328,0.660556,0.665288,0.401321


In [8]:
all_contin, mean_contin = make_aggregated_outputs('continuous_comparison_results_zero_vs_biber', ['pearson', 'spearman', 'MSE', 'RMSE', 'MAE'])

In [9]:
all_contin

informational_vs_involved  non-narrative_vs_narrative  \
outputsTrain pearson                    0.287744                   -0.004086   
             spearman                   0.294482                   -0.031120   
             MSE                        1.424512                    2.008172   
             RMSE                       1.191122                    1.415545   
             MAE                        0.919997                    1.095798   
outputsTest  pearson                    0.281534                   -0.023011   
             spearman                   0.300741                   -0.032063   
             MSE                        1.436932                    2.046021   
             RMSE                       1.195565                    1.428788   
             MAE                        0.908530                    1.093658   
outputsAll   pearson                    0.275746                    0.006148   
             spearman                   0.290048                   -0.013658   
             MSE                        1.448508                    1.987705   
             RMSE                       1.201146                    1.408002   
             MAE                        0.917462                    1.088552   

                       situation-dependent_vs_explicit  \
outputsTrain pearson                          0.031294   
             spearman                         0.076430   
             MSE                              1.937411   
             RMSE                             1.390078   
             MAE                              0.947529   
outputsTest  pearson                          0.025029   
             spearman                         0.057106   
             MSE                              1.949942   
             RMSE                             1.395206   
             MAE                              0.943722   
outputsAll   pearson                          0.035351   
             spearman                         0.085605   
             MSE                              1.929299   
             RMSE                             1.387344   
             MAE                              0.947432   

                       non-persuasive_vs_persuasive  non-abstract_vs_abstract  \
outputsTrain pearson                       0.184369                 -0.003244   
             spearman                      0.137272                 -0.010614   
             MSE                           1.631262                  2.006487   
             RMSE                          1.275083                  1.414781   
             MAE                           1.018886                  1.009200   
outputsTest  pearson                       0.189719                 -0.024917   
             spearman                      0.146376                 -0.027274   
             MSE                           1.620562                  2.049834   
             RMSE                          1.270616                  1.429984   
             MAE                           1.010065                  1.019895   
outputsAll   pearson                       0.192874                 -0.025598   
             spearman                      0.131581                 -0.037557   
             MSE                           1.614252                  2.051197   
             RMSE                          1.269195                  1.430662   
             MAE                           1.018648                  1.011833   

                       compressed_vs_elaborated  
outputsTrain pearson                  -0.006449  
             spearman                 -0.024555  
             MSE                       2.012898  
             RMSE                      1.417110  
             MAE                       1.003340  
outputsTest  pearson                  -0.019309  
             spearman                 -0.036494  
             MSE                       2.038618  
             RMSE                      1.426677  
             MAE

In [10]:
mean_contin

,informational_vs_involved,non-narrative_vs_narrative,situation-dependent_vs_explicit,non-persuasive_vs_persuasive,non-abstract_vs_abstract,compressed_vs_elaborated
MAE,0.915330,1.092670,0.946228,1.015867,1.013643,1.011672
MSE,1.436651,2.013966,1.938884,1.622025,2.035839,2.030477
RMSE,1.195944,1.417445,1.390876,1.271631,1.425143,1.423449
pearson,0.281675,-0.006983,0.030558,0.188987,-0.017920,-0.015239
spearman,0.295090,-0.025614,0.073047,0.138410,-0.025148,-0.034674
